# 股関節DDR 18クラスSegmentation — 前田先生・患者単位3-fold CV

**研究目的:** COCO Segmentation ZIP（健常10名、DDH3名、205フレーム）の `L/R × DS/ST/SW × SL/SM/M` を患者独立で再学習・評価する。

- 各画像は通常6マスク（左右各3領域）。**18クラス＝全データの分類数**で、1画像18マスクではない。
- **3-foldを初期設定**にし、各test foldにDDH1名、健常3〜4名を配置。
- 元ZIPの `train/valid/test` は患者混在のため **評価には一切使用しない**。
- データQC：`R_SD_SL`誤記、左右誤記候補、領域の重複・欠損、患者混入をチェック。自動変更はすべてCSVに記録。左右の高信頼度な入れ替わりと縦長M領域の重複について自動推定修正し、未解決の不整合は学習前に停止。
- 主要評価：未知患者に対する厳密18クラスの**Mask Precision/Recall/mAP50/mAP50-95**、形態だけの**6領域Dice/IoU**。患者単位の集計も出力。
- **注意:** 外側・上内側等の分割の妥当性、歩行相ラベル、DDH診断能力をこのデータだけで確定しない。撮影動画フレームは独立症例ではない。
- 個人情報・DICOM・X線画像・学習重みを公開GitHubへ保存しない。

一般COCO事前学習済み `yolo11n-seg.pt` を初期化に利用。既存股関節モデルの重みは使わない。Ultralyticsの学習・評価部分は**Colab GPUで実行**してください。
## Colab学習後の集計エラー修正（2026-10-26）
学習を完了して `TypeError: agg function failed [how->mean,dtype->object]` が出た場合、モデル学習は成功しています。ランタイムを維持したまま、新しいコードセルに次を貼って実行し、Figure・ZIPセルへ進んでください。

```python
numeric_cols=['n_masks','dice6','iou6','dice18','iou18','detected','phase_correct']
summary=per_patient.groupby('group')[numeric_cols].agg(['mean','std'])
summary.to_csv(OUT_DIR/'OOF_diseasegroup_patientmean_Dice_IoU.csv')
display(per_patient,summary)
```

既に3foldのbest.ptがColabランタイム内にあれば、学習セル7はそれらを自動再利用し再学習しません。Colabが切断されbest.ptが消えた場合は再学習またはバックアップ重みの復元が必要です。


In [ ]:
# Colab環境
%pip -q install "ultralytics>=8.3,<9" pandas pyyaml matplotlib scikit-learn opencv-python-headless

In [ ]:
from pathlib import Path
import os, io, re, gc, json, math, shutil, zipfile, collections, random
import numpy as np, pandas as pd, cv2, yaml, torch
from ultralytics import YOLO
from IPython.display import display

SEED=42
random.seed(SEED); np.random.seed(SEED)
ZIP_PATH = Path('/content/hip segment.coco-segmentation (2).zip')
WORK = Path('/content/maeda_ddr_seg18')
WORK.mkdir(parents=True,exist_ok=True)
RUN_TRAINING = True       # best.ptが存在するfoldは自動再利用（検証は実行）
REUSE_EXISTING_WEIGHTS = True  # 重みがあれば再学習をスキップ
BACKUP_BEST_WEIGHTS = True    # best.ptをZIPでダウンロードし、Colab切断に備える
EPOCHS = 90              # 学会直前なら60-90を目安に調整
IMGSZ = 640
BATCH = 8
PATIENCE = 18
MODEL_INIT = 'yolo11n-seg.pt'
OUT_DIR = WORK/'outputs'; OUT_DIR.mkdir(exist_ok=True)

if not ZIP_PATH.is_file():
    from google.colab import files
    print('COCO Segmentation ZIPをアップロードしてください')
    incoming=files.upload()
    choices=[n for n in incoming if n.lower().endswith('.zip')]
    if len(choices)!=1: raise RuntimeError('研究用Segmentation ZIPを1つ選択してください')
    ZIP_PATH=Path('/content')/choices[0]
print('source:', ZIP_PATH, 'size(MB):', round(ZIP_PATH.stat().st_size/1e6,1))
print('CUDA:', torch.cuda.is_available())

In [ ]:
# 元ZIPの全splitを読み直し、完全なCOCOリストへ統合。画像ごとに患者IDを復元する。
SIDES=('L','R'); PHASES=('DS','ST','SW'); REGIONS=('SL','SM','M')
CLASSES=[f'{side}_{phase}_{region}' for side in SIDES for phase in PHASES for region in REGIONS]
CLASS2ID={s:i for i,s in enumerate(CLASSES)}
PATTERN_NORM=re.compile(r'-(\d{3})\s')
PATTERN_DDH=re.compile(r'DDH[_-]?(\d+)', re.I)

def patient_from_filename(fname):
    m=PATTERN_DDH.search(fname)
    if m: return f'DDH_{int(m.group(1))}'
    m=PATTERN_NORM.search(fname)
    if m:return f'H_{m.group(1)}'
    raise ValueError('患者IDが読み取れません: '+fname)

records=[]; fixes=[]; issue_rows=[]; annotation_rows=[]
with zipfile.ZipFile(ZIP_PATH) as archive:
    for origsplit in ('train','valid','test'):
        obj=json.loads(archive.read(f'{origsplit}/_annotations.coco.json'))
        id2cat={c['id']:c['name'] for c in obj['categories']}
        byimg=collections.defaultdict(list)
        for ann in obj['annotations']:
            byimg[ann['image_id']].append(ann)
        for im in obj['images']:
            fname=im['file_name']; patient=patient_from_filename(fname)
            W,H=int(im['width']),int(im['height'])
            items=[]
            for ann in byimg[im['id']]:
                raw=id2cat[ann['category_id']]; fixed=raw
                if fixed == 'R_SD_SL':
                    fixed='R_DS_SL'; fixes.append({'patient':patient,'file':fname,'from':raw,'to':fixed,'reason':'phase_typo_SD_to_DS'})
                polys=ann.get('segmentation',[])
                # COCO polygon: each part is [x1,y1,x2,y2,..], not COCO RLE
                if not isinstance(polys,list) or not polys or not all(isinstance(part,list) and len(part)>=6 for part in polys):
                    raise ValueError(f'unsupported/empty polygon: {fname}: {raw}')
                xy=np.concatenate([np.asarray(part).reshape(-1,2) for part in polys],axis=0)
                cx=float(xy[:,0].mean())
                # APでは解剖学的Lが画面右に存在する。明確に誤った右ラベルのみ修正する。
                # 中央付近のラベルは医師確認が必要なので自動変更しない。
                if fixed.startswith('R_') and cx/W > 0.70:
                    revised='L_'+fixed[2:]
                    fixes.append({'patient':patient,'file':fname,'from':fixed,'to':revised,'reason':'right_half_R_to_L_high_confidence'})
                    fixed=revised
                if fixed.startswith('L_') and cx/W < 0.30:
                    issue_rows.append({'patient':patient,'file':fname,'issue':'suspected_L_on_R_half','label':fixed,'centroid_ratio':round(cx/W,3)})
                if fixed.startswith('R_') and 0.52 < cx/W <= 0.70:
                    issue_rows.append({'patient':patient,'file':fname,'issue':'check_R_near_midline','label':fixed,'centroid_ratio':round(cx/W,3)})
                if fixed not in CLASS2ID: raise ValueError(f'未定義クラス: {fname}: {raw}->{fixed}')
                items.append({'class':fixed,'polygons':polys})
                annotation_rows.append({'patient':patient,'file':fname,'class':fixed,'original_class':raw,'origsplit':origsplit})
            # 明瞭な重複ラベル修正：SLが2枚・Mが欠損し、縦に長いM領域が一枚ある場合に限定。
            # 推定補正なので修正一覧には残し、学会発表前に画像を確認すること。
            for side in SIDES:
                for phase in PHASES:
                    sl_label=f'{side}_{phase}_SL'; m_label=f'{side}_{phase}_M'
                    sl_items=[a for a in items if a['class']==sl_label]
                    if len(sl_items)==2 and not any(a['class']==m_label for a in items):
                        def geometry_shape(a):
                            q=np.concatenate([np.asarray(poly).reshape(-1,2) for poly in a['polygons']])
                            return (q[:,1].max()-q[:,1].min())/max(q[:,0].max()-q[:,0].min(),1.)
                        candidate=max(sl_items,key=geometry_shape)
                        if geometry_shape(candidate)>1.5:
                            candidate['class']=m_label
                            fixes.append({'patient':patient,'file':fname,'from':sl_label,'to':m_label,
                                          'reason':'duplicated_SL_to_missing_M_vertical_geometry'})
                        else:
                            issue_rows.append({'patient':patient,'file':fname,
                                               'issue':'UNRESOLVED_DUPLICATE_SL','label':sl_label})
            names=[x['class'] for x in items]
            key6=[x[0]+'_'+x.rsplit('_',1)[-1] for x in names]
            if len(items)!=6 or len(set(key6))!=6 or len(set(names))!=6:
                issue_rows.append({'patient':patient,'file':fname,'issue':'NOT_6_UNIQUE_SIDE_REGION','labels':'|'.join(names)})
            records.append({'patient':patient,'group':'DDH' if patient.startswith('DDH_') else 'Healthy','origsplit':origsplit,
                            'filename':fname,'zip_path':f'{origsplit}/{fname}','width':W,'height':H,'anns':items})

assert len(records)==len({r['filename'] for r in records}), '重複画像名：確認が必要'
manifest=pd.DataFrame([{k:v for k,v in r.items() if k!='anns'} for r in records])
anno_df=pd.DataFrame(annotation_rows)
fix_df=pd.DataFrame(fixes); issues_df=pd.DataFrame(issue_rows)
manifest.to_csv(OUT_DIR/'image_manifest.csv',index=False)
anno_df.to_csv(OUT_DIR/'cleaned_annotation_counts.csv',index=False)
fix_df.to_csv(OUT_DIR/'label_auto_corrections_QC.csv',index=False)
issues_df.to_csv(OUT_DIR/'annotation_issues_QC.csv',index=False)
print('Patients:',manifest.groupby('group').patient.nunique().to_dict(),'Frames:',len(manifest))
print('Masks per frame:',len(anno_df)/len(manifest),'18 classes?',anno_df['class'].nunique())
print('automatic fixes:',len(fixes),'QC warnings:',len(issue_rows))
display(anno_df.groupby('class').size().rename('n_masks').reset_index())
if any(x.get('issue')=='NOT_6_UNIQUE_SIDE_REGION' for x in issue_rows):
    print('WARNING: 6領域の不整合があります。医師確認後に学習してください。')
    display(issues_df[issues_df.issue=='NOT_6_UNIQUE_SIDE_REGION'])
    raise ValueError('6領域の不整合が残っています。教師ラベルを確認・補正してから学習してください。')

In [ ]:
# 患者単位3-fold: 各foldにDDH1名 + 健常3〜4名。train/val/testは患者が互いに独立。
patients=manifest[['patient','group']].drop_duplicates()
healthy=sorted(patients.loc[patients.group=='Healthy','patient'])
ddh=sorted(patients.loc[patients.group=='DDH','patient'])
assert len(healthy)==10 and len(ddh)==3, f'期待される症例構成と異なります Healthy={len(healthy)} DDH={len(ddh)}'
rng=np.random.default_rng(SEED); rng.shuffle(healthy); rng.shuffle(ddh)
fold_of={p:i%3 for i,p in enumerate(healthy)}
fold_of.update({p:i for i,p in enumerate(ddh)})
folds=[]
for fold in range(3):
    test_p=sorted([p for p,f in fold_of.items() if f==fold])
    remaining=sorted(set(fold_of)-set(test_p))
    healthy_remaining=sorted([p for p in remaining if p.startswith('H_')])
    vrng=np.random.default_rng(SEED+fold)
    valid_p=sorted(vrng.choice(healthy_remaining,size=2,replace=False).tolist())
    train_p=sorted(set(remaining)-set(valid_p))
    assert not(set(train_p)&set(valid_p) or set(train_p)&set(test_p) or set(valid_p)&set(test_p))
    assert len([p for p in test_p if p.startswith('DDH_')])==1
    folds.append({'fold':fold+1,'train':train_p,'val':valid_p,'test':test_p})
rows=[]
for f in folds:
    for stage in ['train','val','test']:
        for p in f[stage]:
            rows.append({'fold':f['fold'],'stage':stage,'patient':p,'group':'DDH' if p.startswith('DDH_') else 'Healthy',
                         'n_frames':int((manifest.patient==p).sum())})
split_df=pd.DataFrame(rows)
split_df.to_csv(OUT_DIR/'patient_cv_split_manifest.csv',index=False)
display(split_df.pivot_table(index=['fold','stage'],columns='group',values='patient',aggfunc='count',fill_value=0))

In [ ]:
# COCO Polygon -> YOLO segmentation (18 classes): normalized contours, preserving 6 masks/image
# モデル評価に使う原画像のコピーはColabローカルだけに置き、GitHubには載せない。
def polygon_to_line(item,W,H):
    normparts=[]
    for part in item['polygons']:
        pts=np.asarray(part,dtype=float).reshape(-1,2)
        pts[:,0]=np.clip(pts[:,0]/W,0,1)
        pts[:,1]=np.clip(pts[:,1]/H,0,1)
        if len(pts)>=3:normparts.append(pts)
    # COCO 1 annotation can technically contain several polygons; current input has one.
    if len(normparts)!=1: raise ValueError('複数partマスクです。COCO polygon->YOLO統合処理を要追加')
    a=normparts[0].ravel()
    return str(CLASS2ID[item['class']])+' '+' '.join(f'{v:.6f}' for v in a)+'\n'

# 各foldに書き出す。大量の画像は同一のローカルキャッシュからcopyする。
cache=WORK/'original_images';cache.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    for i,r in enumerate(records):
        target=cache/r['filename']
        if not target.exists(): target.write_bytes(z.read(r['zip_path']))
        r['source_path']=target
rec_by_name={r['filename']:r for r in records}

def materialize_fold(f):
    root=WORK/f"fold_{f['fold']}"
    for stage in ('train','val','test'):
        im_dir=root/'images'/stage; lb_dir=root/'labels'/stage
        im_dir.mkdir(parents=True,exist_ok=True);lb_dir.mkdir(parents=True,exist_ok=True)
        for r in records:
            if r['patient'] not in f[stage]:continue
            dst=im_dir/r['filename']
            if not dst.exists(): shutil.copy2(r['source_path'],dst)
            lines=''.join(polygon_to_line(a,r['width'],r['height']) for a in r['anns'])
            (lb_dir/(Path(r['filename']).stem+'.txt')).write_text(lines,encoding='utf-8')
    y={'path':str(root),'train':'images/train','val':'images/val','test':'images/test',
       'names':{i:c for i,c in enumerate(CLASSES)}}
    yp=root/'data.yaml'; yp.write_text(yaml.safe_dump(y,allow_unicode=True,sort_keys=False))
    return root,yp

foldpaths=[materialize_fold(f) for f in folds]
for f,(root,yp) in zip(folds,foldpaths):
    print('fold',f['fold'], 'train',len(list((root/'images/train').glob('*.jpg'))),
          'val',len(list((root/'images/val').glob('*.jpg'))),
          'test',len(list((root/'images/test').glob('*.jpg'))))

In [ ]:
# データQC可視化：左右別・領域別のマスク6個が正しいか画面上で確認
import matplotlib.pyplot as plt

def draw_gt(image_rec):
    raw=cv2.imread(str(image_rec['source_path']));rgb=cv2.cvtColor(raw,cv2.COLOR_BGR2RGB)
    canvas=rgb.copy();rng=np.random.default_rng(15)
    for a in image_rec['anns']:
        color=(rng.integers(70,255,size=3)).astype(np.uint8)
        for poly in a['polygons']:
            xy=np.rint(np.asarray(poly).reshape(-1,2)).astype(np.int32)
            cv2.fillPoly(canvas,[xy],tuple(int(x) for x in color))
    return cv2.addWeighted(rgb,.65,canvas,.35,0)

sample=[next(r for r in records if r['patient']==p) for p in ('H_005','DDH_1')]
fig,axes=plt.subplots(1,2,figsize=(12,6))
for ax,r in zip(axes,sample):
    ax.imshow(draw_gt(r));ax.set_title(r['patient']+' : '+r['filename'][:30]);ax.axis('off')
plt.tight_layout();plt.show()

In [ ]:
# 3-fold 学習＋未知患者testだけで評価。validationは別患者でearly stoppingに使用。
# WARNING: Colab GPUが必要です。RUN_TRAINING=Falseならこのセルはスキップ。
metrics_rows=[];best_weights={}
if RUN_TRAINING:
    assert torch.cuda.is_available(),'GPUランタイムに切り替えてください'
    for f,(root,yfile) in zip(folds,foldpaths):
        print('=== FOLD',f['fold'],'Test patients:',f['test'],'===')
        best=WORK/'runs'/f"fold_{f['fold']}"/'weights'/'best.pt'
        if REUSE_EXISTING_WEIGHTS and best.is_file():
            print(f'[REUSE] fold {f["fold"]}: {best} を利用し再学習をスキップ')
        else:
            train_model=YOLO(MODEL_INIT)
            train_model.train(data=str(yfile),task='segment',epochs=EPOCHS,imgsz=IMGSZ,batch=BATCH,
                              patience=PATIENCE,device=0,seed=SEED+f['fold'],workers=2,cache=False,
                              fliplr=0.0,flipud=0.0,mosaic=0.0,close_mosaic=0,
                              degrees=0.0,translate=0.04,scale=0.10,
                              project=str(WORK/'runs'),name=f"fold_{f['fold']}",exist_ok=True,plots=False,save=True)
            del train_model;gc.collect();torch.cuda.empty_cache()
        assert best.is_file(),f'best.pt missing {best}'
        best_weights[f['fold']]=str(best)
        model=YOLO(str(best))
        test_metrics=model.val(data=str(yfile),split='test',imgsz=IMGSZ,batch=BATCH,device=0,plots=False,
                               project=str(WORK/'runs'),name=f"fold_{f['fold']}_test",exist_ok=True)
        metrics_rows.append({'fold':f['fold'],'test_patients':'|'.join(f['test']),
          'n_test_images':len(list((root/'images/test').glob('*.jpg'))),
          'mask_precision':float(test_metrics.seg.mp),'mask_recall':float(test_metrics.seg.mr),
          'mask_mAP50':float(test_metrics.seg.map50),'mask_mAP50_95':float(test_metrics.seg.map),
          'box_precision':float(test_metrics.box.mp),'box_recall':float(test_metrics.box.mr),
          'box_mAP50':float(test_metrics.box.map50),'box_mAP50_95':float(test_metrics.box.map)})
        pd.DataFrame(metrics_rows).to_csv(OUT_DIR/'CV_3fold_18class_mask_box_metrics.csv',index=False)
        print(metrics_rows[-1])
        del model,test_metrics;gc.collect();torch.cuda.empty_cache()
else:
    print('RUN_TRAINING=False: QC・split生成のみ実行')
if metrics_rows:
    mt=pd.DataFrame(metrics_rows)
    sm=mt.select_dtypes('number').drop(columns=['fold','n_test_images']).agg(['mean','std']).T
    sm.to_csv(OUT_DIR/'CV_3fold_mean_SD_18class.csv')
    display(sm)
# Colab runtimeが切れても再学習を回避できるよう、研究用重み3foldをローカルに保存。
if RUN_TRAINING and BACKUP_BEST_WEIGHTS and best_weights:
    weight_bundle=Path('/content/Maeda_HipDDR_18class_fold_best_weights.zip')
    with zipfile.ZipFile(weight_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for fold,pt in sorted(best_weights.items()):
            z.write(pt, arcname=f'fold_{fold}/best.pt')
    print('Saved model backup:', weight_bundle)
    try:
        from google.colab import files
        files.download(str(weight_bundle))
    except ImportError:
        pass


In [ ]:
# 未知患者に対する6領域Dice/IoUと、18クラス一致を要するDiceを同時評価。
# 6領域: phase(SW/ST/DS)を無視して side/region の輪郭を評価。
def make_binary_mask(polylist, H,W):
    m=np.zeros((H,W),dtype=np.uint8)
    for xyflat in polylist:
        xy=np.rint(np.asarray(xyflat,dtype=float).reshape(-1,2)).astype(np.int32)
        cv2.fillPoly(m,[xy],1)
    return m

def poly_from_prediction(xy,H,W):
    m=np.zeros((H,W),dtype=np.uint8)
    ar=np.asarray(xy,dtype=float)
    if len(ar)>=3: cv2.fillPoly(m,[np.rint(ar).astype(np.int32)],1)
    return m

def scores(gt,pr):
    intersection=int(np.count_nonzero(gt&pr)); a=int(gt.sum()); b=int(pr.sum())
    return (2*intersection/(a+b) if a+b else float('nan'),
            intersection/(a+b-intersection) if a+b-intersection else float('nan'))

mask_results=[]
if RUN_TRAINING:
    for f,(root,yfile) in zip(folds,foldpaths):
        model=YOLO(best_weights[f['fold']])
        for r in records:
            if r['patient'] not in f['test']:continue
            H,W=r['height'],r['width']
            predictions=model.predict(source=str(root/'images/test'/r['filename']),imgsz=IMGSZ,
                                      device=0,conf=0.15,iou=0.6,retina_masks=True,verbose=False)
            pr=predictions[0]
            chosen={}
            if pr.masks is not None and pr.boxes is not None:
                for idx,(c,conf) in enumerate(zip(pr.boxes.cls.cpu().numpy().astype(int),
                                                   pr.boxes.conf.cpu().numpy())):
                    label=CLASSES[c]; parts=label.split('_'); geo=parts[0]+'_'+parts[2]
                    if geo not in chosen or conf > chosen[geo][0]:
                        chosen[geo]=(float(conf),label,poly_from_prediction(pr.masks.xy[idx],H,W))
            for gt in r['anns']:
                gtlabel=gt['class']; side,phase,reg=gtlabel.split('_');geo=f'{side}_{reg}'
                target=make_binary_mask(gt['polygons'],H,W)
                chosen_pr=chosen.get(geo)
                if chosen_pr is None:
                    predlabel=None; mask=np.zeros_like(target)
                else:
                    _,predlabel,mask=chosen_pr
                dice6,iou6=scores(target,mask)
                strict_mask=mask if predlabel==gtlabel else np.zeros_like(target)
                dice18,iou18=scores(target,strict_mask)
                mask_results.append({'fold':f['fold'],'patient':r['patient'],'group':r['group'],
                                     'filename':r['filename'],'side':side,'region':reg,'GT_phase':phase,
                                     'GT_class':gtlabel,'pred_class':predlabel,
                                     'dice6':dice6,'iou6':iou6,'dice18':dice18,'iou18':iou18,
                                     'phase_correct':int(predlabel==gtlabel),'detected':int(predlabel is not None)})
        del model;gc.collect();torch.cuda.empty_cache()

    md=pd.DataFrame(mask_results)
    md.to_csv(OUT_DIR/'OOF_mask_dice_iou_6vs18_detail.csv',index=False)
    per_patient=md.groupby(['patient','group'],as_index=False).agg(
        n_masks=('dice6','size'),dice6=('dice6','mean'),iou6=('iou6','mean'),
        dice18=('dice18','mean'),iou18=('iou18','mean'),detected=('detected','mean'),phase_correct=('phase_correct','mean'))
    per_patient.to_csv(OUT_DIR/'OOF_patient_Dice_IoU.csv',index=False)
    numeric_cols=['n_masks','dice6','iou6','dice18','iou18','detected','phase_correct']
    summary=per_patient.groupby('group')[numeric_cols].agg(['mean','std'])
    summary.to_csv(OUT_DIR/'OOF_diseasegroup_patientmean_Dice_IoU.csv')
    display(per_patient)
    print('Patient-macro Dice6:',round(per_patient.dice6.mean(),3),'Patient-macro Dice18:',round(per_patient.dice18.mean(),3))
else:print('学習が未実行です')

In [ ]:
# 例示Figureは「患者を学習から除外したモデル」の予測のみ利用
if RUN_TRAINING:
    from PIL import Image
    sample_patients=['H_005','DDH_1']
    fig,axes=plt.subplots(2,2,figsize=(13,13))
    for row,pat in enumerate(sample_patients):
        r=next(x for x in records if x['patient']==pat)
        fid=fold_of[pat]+1
        predmodel=YOLO(best_weights[fid])
        gt_img=draw_gt(r)
        result=predmodel.predict(str(r['source_path']),imgsz=IMGSZ,conf=0.15,retina_masks=True,device=0,verbose=False)[0]
        pred_rgb=cv2.cvtColor(result.plot(),cv2.COLOR_BGR2RGB)
        axes[row,0].imshow(gt_img);axes[row,0].set_title(f'{pat} / 手動GT (fold{fid}の未知患者)')
        axes[row,1].imshow(pred_rgb);axes[row,1].set_title(f'{pat} / AI予測 (18-class)')
        for col in [0,1]:axes[row,col].axis('off')
        del predmodel;gc.collect();torch.cuda.empty_cache()
    plt.tight_layout()
    fig.savefig(OUT_DIR/'Figure_GT_vs_AI_unseen_patients.png',dpi=180,bbox_inches='tight')
    plt.show()

In [ ]:
# 結果のCSV・FigをZIPにまとめる（学習済みweightや患者画像を外部公開しない）
import zipfile
bundle=Path('/content/Maeda_HipDDR_18class_Group3Fold_results.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as out:
    for fp in sorted(OUT_DIR.rglob('*')):
        if fp.is_file():out.write(fp,fp.relative_to(OUT_DIR))
print('Saved:',bundle,'size MB:',round(bundle.stat().st_size/1e6,2))
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:pass

## 発表・解釈の注意

1. **18クラスmask mAP** は「位置・形状・左右・歩行相のすべてを正しく予測したか」。**6領域Dice** は「左右の3解剖領域の輪郭を正しく抽出できたか（歩行相は問わない）」を別々に示します。
2. 3-fold testは**患者単位で独立**。健常10例、DDH3例であり、フレーム数を患者数として統計的に扱いません。
3. 本ノートではモデルの「歩行相推定」が動画全体の時間情報なしに成立するかの確証は得られません。将来は、**6解剖クラスSegmentation + 左右別歩行相分類**という二段階設計と比較する価値があります。
4. `DS/ST/SW` の定義や `SL/SM/M` の解剖学的境界は前田先生との最終確認が必要です。
5. **GPUでの学習・mAP/Dice算出が終了するまで、結果数値を学会資料に記載しないでください。**

参考： [Ultralytics Segmentation Task](https://docs.ultralytics.com/tasks/segment/), [Segmentation Dataset Format](https://docs.ultralytics.com/datasets/segment/)。